In [2]:
import torch
import pickle
from torch import nn
from tqdm import tqdm
from MyDataLoader import *
from animation_marking import animation_double

In [3]:
class LSTM(nn.Module):
    def __init__(self,input_size,hidden_size,linear_size,output_size,train_steps=20):
        super().__init__()
        
        self.rnn=nn.LSTM(input_size,hidden_size,num_layers=3,batch_first=True,dropout=0.1)
        self.linear=nn.Sequential(nn.Dropout(0.1),
            nn.Linear(hidden_size, linear_size),
            nn.BatchNorm1d(linear_size), 
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(linear_size, output_size)
        )
        
        self.hidden_size=hidden_size
        
        # self.loss_scale = torch.arange(1,train_steps+1).reshape(-1,1).float()
    
    def forward(self,x,state):
        y_hat,new_state=self.rnn(x,state)
        y_hat=y_hat.reshape(-1,y_hat.shape[-1])
        y_hat=self.linear(y_hat)
        
        if isinstance(new_state,tuple):
            new_state=tuple(item.detach() for item in new_state)
        else:
            new_state=new_state.detach()
        return y_hat,new_state


In [4]:
with open(r"C:\Users\34362\Desktop\LSTM\results\circular.pkl","rb")as f:
    circular=pickle.load(f)
with open(r"C:\Users\34362\Desktop\LSTM\results\infinity_like.pkl","rb")as f:
    infinity_like=pickle.load(f)

In [5]:
batch_size,train_steps,input_size,hidden_size,linear_size,output_size,lr=64,20,3,300,168,3,0.01
pred_steps=15
test_num=10
circular_norm=normalize(circular)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_loder=MyDataLoader(circular_norm,0.8,10,batch_size,train_steps,pred_steps)
net=LSTM(input_size,hidden_size,linear_size,output_size).to(device)


In [6]:
# net = torch.load("model.pth", map_location=device, weights_only=False)
net.load_state_dict(
    torch.load("time_seq_best.opt", map_location=device, weights_only=False)
)

<All keys matched successfully>

In [7]:
test_loder=data_loder.test_iter()
mutil_steps_pred=pred_time_seq(test_loder,net,device,10,train_steps,hidden_size,pred_steps)
mutil_steps_pred=denorm_and_pad(mutil_steps_pred,pred_steps,train_steps,input_size)
print(mutil_steps_pred[0].shape)

Multi step predict:   0%|          | 0/15 [00:00<?, ?it/s]

Multi step predict: 100%|██████████| 15/15 [00:00<00:00, 22.79it/s]

[ 0.06660479  0.15352495 12.16919512]
[23.04761502 22.93268723  4.43137723]
(106, 16, 3)


In [8]:
mutil_steps_pred[5][20]

array([[-37.18280429,  -7.8636561 ,   9.61131176],
       [-37.17282541,  -7.89552799,   9.71494818],
       [-37.25670353,  -7.74208108,   9.84087379],
       [-37.44425543,  -7.45248188,   9.99337729],
       [-37.71850711,  -7.05763044,  10.17587177],
       [-38.06944673,  -6.58645091,  10.39081259],
       [-38.48253183,  -6.05896352,  10.6408194 ],
       [-38.94491513,  -5.49119682,  10.92816266],
       [-39.44396368,  -4.89563636,  11.25486654],
       [-39.96727257,  -4.2819092 ,  11.62242153],
       [-40.50201102,  -3.65716076,  12.03152064],
       [-41.03455147,  -3.026032  ,  12.48174192],
       [-41.55042016,  -2.39044221,  12.97125436],
       [-41.86742004,  -1.69631036,  13.48783181],
       [-41.96733251,  -0.84239477,  14.01403213],
       [-41.97436059,   0.190881  ,  14.54320603]])

In [ ]:
circular[3278+5][19:42]

,time,tx,ty,tz
19,1.699986e+09,-37.182804,-7.863656,9.611312
20,1.699986e+09,-36.991504,-8.097185,9.699399
21,1.699986e+09,-36.777091,-8.293594,9.775260
22,1.699986e+09,-36.560882,-8.441696,9.831011
23,1.699986e+09,-36.323924,-8.554458,9.872236
24,1.699986e+09,-36.084605,-8.617348,9.895795
25,1.699986e+09,-35.829942,-8.639403,9.910849
26,1.699986e+09,-35.579737,-8.624170,9.911345
27,1.699986e+09,-35.327755,-8.566529,9.904113
28,1.699986e+09,-35.079541,-8.470066,9.892291


: 

In [ ]:
test_start_idx = len(circular) - test_num
for i in range(test_num):
    dataset1=circular[i+test_start_idx]
    dataset2=mutil_steps_pred[i]
    animation_double(i,dataset1,dataset2)